### modelo predictivo para la deteccion de paros cardiacos

### 1. Carga y exploracion inicial del data frame

In [8]:
# Librerias de visualizacion y importacion de datos.abs
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 

df = pd.read_csv("../data/raw/HeartAttackDataSet.csv")

# Dimenciones del data frame 
print(f"=" * 80)
print(f"Dimensiones del data frame: {df.shape[0]} numero de filas y {df.shape[1]} numero de columnas ")
# Información del data frame
print(f"=" * 80)
print (f"Tipos de datos del data frame")
print(f"=" * 80)




Dimensiones del data frame: 303 numero de filas y 14 numero de columnas 
Tipos de datos del data frame


In [9]:
# Primeras 5 filas del data frame
print(f"Primeras 5 filas del data frame")
df.head()
# Validacion de tipos de datos del data frame 

Primeras 5 filas del data frame


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


### 2. Analisis de calidad de datos.
#### 2.1 Analisis de datos faltantes (nulos) y duplicados.
- La integridad de los datos previene sesgos algoritmicos, al tratarse de un modelo orientado a la salud, tenemos que tener la mejor calida de datos posibles.

In [10]:
# Valores nulos.
print(f"=" *80)
print(f"total de nulos: {df.isnull().sum()}")
# Valores duplicados.
print(f"=" *80)
duplicated = df.duplicated().sum()
print(f"Valores duplicados encontrados: {duplicated}")

if duplicated > 0:
    print(f"=" *80)
    print(f"Columna(s) donde se encuentra el valor duplicado") # Al tratarse del sector salud, siempre me interesara saber que dato es el que esta duplicado.
    display(df[df.duplicated()])

total de nulos: age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64
Valores duplicados encontrados: 1
Columna(s) donde se encuentra el valor duplicado


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
164,38,1,2,138,175,0,1,173,0,0.0,2,4,2,1


**Interpretación: Observamos que el data set no cuenta con valores nulos, y encontramos un valor duplicado en la columna 164, al tratarse de un data set pequeño, la mejor opcion sera eliminar el duplicado.

In [11]:
# Eliminar duplicado

df.drop_duplicates()
print(f"Valores duplicados despues de limpieza: {df.duplicated().sum()}")

Valores duplicados despues de limpieza: 1


### 3. Estadisticas descriptivas y target
#### 3.1 Analisis de la variable objetivo (target) 
Fundamentacion: En problemas de clasificacion un desbalance de clases podria invalidad la metrica accuracy

In [12]:
# Muestro estadisticas descriptiva la cual nos proporciona un resumen de los datos como media, desviación, etc.
numeric_stats = df.describe().T
numeric_stats['cv'] = numeric_stats['std'] / numeric_stats['mean'] * 100 # oeficiente de variación
numeric_stats['range'] = numeric_stats['max'] - numeric_stats['min']
numeric_stats['iqr'] = numeric_stats['75%'] - numeric_stats['25%']

# Interpretacion de coeficiente de variacion
print(f"="*80)
display(numeric_stats)
print(f"="* 80)
print(f"Interpretación de coeficiente de variacion: ")
print(f"="*80)
print(f"CV < 15%: Baja variabilidad")
print(f"CV 15% - 35%: Variabilidad moderada")
print(f"CV > 35%: Alta variabilidad")
high_var = numeric_stats[numeric_stats['cv'] > 35]['cv'].sort_values(ascending=False)
for i, cv in high_var.items():
    print(f"- {i}: {cv:.2f}%")
print("Total de variables con CV alto > 35%: ", high_var.value_counts().sum())

,count,mean,std,min,25%,50%,75%,max,cv,range,iqr
age,303.0,54.366337,9.082101,29.0,47.5,55.0,61.0,77.0,16.705376,48.0,13.5
sex,303.0,0.683168,0.466011,0.0,0.0,1.0,1.0,1.0,68.213178,1.0,1.0
cp,303.0,0.966997,1.032052,0.0,0.0,1.0,2.0,3.0,106.727612,3.0,2.0
trestbps,303.0,131.623762,17.538143,94.0,120.0,130.0,140.0,200.0,13.324450,106.0,20.0
chol,303.0,246.264026,51.830751,126.0,211.0,240.0,274.5,564.0,21.046822,438.0,63.5
fbs,303.0,0.148515,0.356198,0.0,0.0,0.0,0.0,1.0,239.839902,1.0,0.0
restecg,303.0,0.528053,0.525860,0.0,0.0,1.0,1.0,2.0,99.584661,2.0,1.0
thalach,303.0,149.646865,22.905161,71.0,133.5,153.0,166.0,202.0,15.306142,131.0,32.5
exang,303.0,0.326733,0.469794,0.0,0.0,0.0,1.0,1.0,143.785579,1.0,1.0
oldpeak,303.0,1.039604,1.161075,0.0,0.0,0.8,1.6,6.2,111.684359,6.2,1.6


Interpretación de coeficiente de variacion: 
CV < 15%: Baja variabilidad
CV 15% - 35%: Variabilidad moderada
CV > 35%: Alta variabilidad
- fbs: 239.84%
- exang: 143.79%
- ca: 140.20%
- oldpeak: 111.68%
- cp: 106.73%
- restecg: 99.58%
- target: 91.60%
- sex: 68.21%
- slope: 44.04%
Total de variables con CV alto > 35%:  9


Podemos observar que 9 varibles tienen alto coeficiente de variacion lo cual indica precencia de outliers, el coeficiente de variacion es una medida de dispersion que mide la variabilidad de una variable en relacion a su media. Lo cual nos sirve para poder identificar que tipo de escalador debemos utilizar en el preprocesamiento de datos.
#### En este caso el escalador sugerido es StandardScaler 

# 4. Distribucion de variable objetivo


In [19]:
target_counts = df['target'].value_counts()
target_pct = df['target'].value_counts(normalize=True) * 100
target_summary = pd.DataFrame({
    'Clase': ['Sin enfermedad (0)', 'Con enfermedad (1)'],
    'Frecuencia': target_counts.values,
    'Porcentaje': target_pct
})
display(target_summary)

# Balance ratio
balance_ratio = target_counts.min() / target_counts.max()
print(f"Balance ratio {balance_ratio:.3f}")

,Clase,Frecuencia,Porcentaje
target,,,
1,Sin enfermedad (0),165,54.455446
0,Con enfermedad (1),138,45.544554


Balance ratio 0.836


A simple vista podemos detectar un data set bastante balanceado, es decir, que la proporción de datos de paro cardiaco es similar a la proporción de datos de no paro cardiaco.